# Stack Overflow Tag Prediction

**Goal:** predict a small set of relevant Stack Overflow tags from a question title and body.

```text
project-root/
├── data/
│   ├── raw/
│   └── processed/
├── models/
├── notebooks/
└── backend/
```

In [ ]:
# Core libraries
from pathlib import Path
from collections import Counter
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    jaccard_score,
)

warnings.filterwarnings("ignore")
plt.style.use("default")
sns.set_theme(style="whitegrid")

In [ ]:
# Project paths
# Keep the raw data files under data/raw/ and write processed files to data/processed/.
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

QUESTIONS_FILE = RAW_DIR / "Questions.csv.zip"
TAGS_FILE = RAW_DIR / "Tags.csv.zip"
MERGED_OUTPUT = PROCESSED_DIR / "stack_overflow_questions_tags.csv"
CLEAN_OUTPUT = PROCESSED_DIR / "stack_overflow_questions_tags_clean.csv"

MIN_SCORE = 1
TOP_N_TAGS = 50
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_FEATURES = 50000

## 1. Load and merge the raw Stack Overflow data

In [ ]:
def load_stackoverflow_data(questions_path: Path, tags_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not questions_path.exists():
        raise FileNotFoundError(f"Questions file not found: {questions_path}")
    if not tags_path.exists():
        raise FileNotFoundError(f"Tags file not found: {tags_path}")

    questions = pd.read_csv(questions_path, encoding="ISO-8859-1")
    tags = pd.read_csv(tags_path, encoding="ISO-8859-1", dtype={"Tag": str})
    return questions, tags


def merge_questions_and_tags(questions: pd.DataFrame, tags: pd.DataFrame) -> pd.DataFrame:
    questions = questions.copy()
    tags = tags.copy()

    # Remove columns that are not required for tag prediction.
    columns_to_drop = [c for c in ["OwnerUserId", "CreationDate", "ClosedDate"] if c in questions.columns]
    questions = questions.drop(columns=columns_to_drop, errors="ignore")

    tags["Tag"] = tags["Tag"].astype(str)
    grouped_tags = tags.groupby("Id")["Tag"].apply(list).reset_index()
    grouped_tags["Tags"] = grouped_tags["Tag"].apply(lambda items: " ".join(items))
    grouped_tags = grouped_tags.drop(columns=["Tag"])

    merged = questions.merge(grouped_tags, on="Id", how="inner")
    merged = merged[merged["Score"] > MIN_SCORE].copy()
    merged = merged.drop_duplicates(subset=["Title", "Body", "Tags"])
    merged = merged.drop(columns=[c for c in ["Id", "Score"] if c in merged.columns], errors="ignore")
    return merged


questions_df, tags_df = load_stackoverflow_data(QUESTIONS_FILE, TAGS_FILE)
raw_merged_df = merge_questions_and_tags(questions_df, tags_df)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
raw_merged_df.to_csv(MERGED_OUTPUT, index=False)

raw_merged_df.head()

## 2. Explore the tag distribution

In [ ]:
# Split tags into lists for analysis.
explore_df = raw_merged_df.copy()
explore_df["Tags"] = explore_df["Tags"].apply(lambda x: x.split())

all_tags = [tag for row in explore_df["Tags"] for tag in row]
tag_counts = Counter(all_tags)

tag_freq_df = (
    pd.DataFrame(tag_counts.items(), columns=["Tag", "Count"])
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 5))
tag_freq_df.head(20).plot(kind="bar", x="Tag", y="Count", ax=ax, legend=False)
ax.set_title("Top 20 tags")
ax.set_xlabel("Tag")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

## 3. Text preprocessing

In [ ]:
def clean_html(text: str) -> str:
    return BeautifulSoup(str(text), "html.parser").get_text(" ", strip=True)


def preprocess_question(text: str) -> str:
    text = clean_html(text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9+#\. ]", " ", text)  # keep code-like tokens such as c++, c#, .net
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [tok for tok in text.split() if tok not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)


clean_df = raw_merged_df.copy()
clean_df["question_text"] = (
    clean_df["Title"].fillna("").astype(str) + " " + clean_df["Body"].fillna("").astype(str)
)
clean_df["question_text"] = clean_df["question_text"].apply(preprocess_question)

clean_df = clean_df[clean_df["question_text"].str.len() > 0].copy()
clean_df.to_csv(CLEAN_OUTPUT, index=False)

clean_df[["Title", "question_text", "Tags"]].head()

## 4. Prepare multilabel targets

In [ ]:
# Keep only the most common tags to reduce label noise and class sparsity.
tag_counter = Counter(tag for tags in raw_merged_df["Tags"].str.split() for tag in tags)
top_tags = [tag for tag, _ in tag_counter.most_common(TOP_N_TAGS)]

label_df = clean_df.copy()
label_df["Tags"] = label_df["Tags"].str.split()
label_df["Tags"] = label_df["Tags"].apply(lambda tags: [tag for tag in tags if tag in top_tags])
label_df = label_df[label_df["Tags"].map(bool)].copy()

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(label_df["Tags"])
X_text = label_df["question_text"].tolist()

print("Samples:", len(X_text))
print("Labels:", len(mlb.classes_))
print("Top tags:", list(mlb.classes_[:10]))

## 5. Feature extraction

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=2,
    token_pattern=r"(?u)\b\w+\b",
)

X = vectorizer.fit_transform(X_text)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 6. Train a strong baseline model

In [ ]:
def evaluate_multilabel(y_true, y_pred) -> dict:
    return {
        "jaccard_samples": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


baseline_clf = OneVsRestClassifier(LinearSVC())
baseline_clf.fit(X_train, y_train)
y_pred = baseline_clf.predict(X_test)

scores = evaluate_multilabel(y_test, y_pred)
pd.Series(scores).to_frame("Score")

In [ ]:
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))

## 7. Predict the top-k tags

In [ ]:
def predict_top_k_tags(model, X_matrix, mlb: MultiLabelBinarizer, k: int = 5):
    scores = model.decision_function(X_matrix)
    if scores.ndim == 1:
        scores = scores.reshape(-1, 1)

    top_indices = np.argsort(scores, axis=1)[:, -k:]
    top_labels = []
    for row in top_indices:
        # Reverse so the highest score appears first.
        top_labels.append([mlb.classes_[idx] for idx in row[::-1]])
    return top_labels


sample_top5 = predict_top_k_tags(baseline_clf, X_test[:3], mlb, k=5)
sample_top5

## 8. Suggested next step

When the trained model is stable, serialize the vectorizer, multilabel binarizer, and classifier with `joblib` or `pickle`, then load them inside your backend API to serve tag predictions for the website.